In [ ]:
%%sql

-- ============================================================
-- GOLD FIRE RISK
-- ============================================================

-- ------------------------------------------------------------
-- 1. Crear tabla Gold si todavía no existe
-- ------------------------------------------------------------

CREATE TABLE IF NOT EXISTS gold_fire_risk (
    fire_id STRING,
    fire_detection_timestamp TIMESTAMP,

    fire_latitude DOUBLE,
    fire_longitude DOUBLE,

    forecast_timestamp TIMESTAMP,

    weather_latitude DOUBLE,
    weather_longitude DOUBLE,
    weather_distance_km DOUBLE,

    temperature_2m DOUBLE,
    relative_humidity_2m DOUBLE,
    precipitation DOUBLE,
    wind_speed_10m DOUBLE,

    brightness DOUBLE,
    frp DOUBLE,

    temperature_score INT,
    humidity_score INT,
    wind_score INT,
    precipitation_score INT,
    fire_intensity_score INT,

    fire_risk_score INT,
    fire_risk_level STRING,

    source_fire STRING,
    source_weather STRING,

    updated_at TIMESTAMP
);


-- ------------------------------------------------------------
-- 2. Incendios NASA operativos (Últimas 24h)
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW recent_fires AS
SELECT
    CAST(
        CONCAT(
            CAST(latitude AS STRING),
            '_',
            CAST(longitude AS STRING),
            '_',
            CAST(fire_detection_timestamp AS STRING)
        )
        AS STRING
    ) AS fire_id,

    fire_detection_timestamp,

    CAST(latitude AS DOUBLE) AS fire_latitude,
    CAST(longitude AS DOUBLE) AS fire_longitude,

    -- Mapeo de columnas corregido desde Silver
    CAST(bright_ti4 AS DOUBLE) AS brightness,
    CAST(fire_radiative_power AS DOUBLE) AS frp,

    landing_source_file AS source_fire

FROM silver_nasa_fires

WHERE fire_detection_timestamp >= current_timestamp() - INTERVAL 24 HOURS
  AND fire_detection_timestamp <= current_timestamp();


-- ------------------------------------------------------------
-- 3. Weather disponible desde ahora hacia el futuro
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW future_weather AS
SELECT
    CAST(latitude AS DOUBLE) AS weather_latitude,
    CAST(longitude AS DOUBLE) AS weather_longitude,

    forecast_timestamp,

    -- Mapeo de columnas corregido desde Silver
    CAST(temperature_celsius AS DOUBLE) AS temperature_2m,
    CAST(humidity_percentage AS DOUBLE) AS relative_humidity_2m,
    CAST(precipitation_mm AS DOUBLE) AS precipitation,
    CAST(wind_speed_kmh AS DOUBLE) AS wind_speed_10m,

    ingestion_timestamp,
    landing_source_file AS source_weather

FROM silver_weather

WHERE forecast_timestamp >= current_timestamp();


-- ------------------------------------------------------------
-- 4. Cruce espacial (Radio 25 km)
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW weather_candidates AS
SELECT
    f.fire_id,
    f.fire_detection_timestamp,

    f.fire_latitude,
    f.fire_longitude,

    f.brightness,
    f.frp,

    f.source_fire,

    w.weather_latitude,
    w.weather_longitude,

    w.forecast_timestamp,

    w.temperature_2m,
    w.relative_humidity_2m,
    w.precipitation,
    w.wind_speed_10m,

    w.source_weather,
    w.ingestion_timestamp AS weather_ingestion_timestamp,

    (
        6371.0 * 2.0 * ASIN(
            SQRT(
                POWER(SIN(RADIANS(w.weather_latitude - f.fire_latitude) / 2.0), 2)
                + COS(RADIANS(f.fire_latitude)) * COS(RADIANS(w.weather_latitude))
                * POWER(SIN(RADIANS(w.weather_longitude - f.fire_longitude) / 2.0), 2)
            )
        )
    ) AS weather_distance_km

FROM recent_fires f
INNER JOIN future_weather w
    ON w.weather_latitude BETWEEN f.fire_latitude - 0.25 AND f.fire_latitude + 0.25
   AND w.weather_longitude BETWEEN f.fire_longitude - 0.35 AND f.fire_longitude + 0.35
WHERE
    (
        6371.0 * 2.0 * ASIN(
            SQRT(
                POWER(SIN(RADIANS(w.weather_latitude - f.fire_latitude) / 2.0), 2)
                + COS(RADIANS(f.fire_latitude)) * COS(RADIANS(w.weather_latitude))
                * POWER(SIN(RADIANS(w.weather_longitude - f.fire_longitude) / 2.0), 2)
            )
        )
    ) <= 25.0;


-- ------------------------------------------------------------
-- 5. Seleccionar Weather más cercano
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW nearest_weather AS
SELECT *
FROM (
    SELECT
        wc.*,
        ROW_NUMBER() OVER (
            PARTITION BY fire_id, fire_detection_timestamp, forecast_timestamp
            ORDER BY weather_distance_km ASC, weather_ingestion_timestamp DESC
        ) AS rn
    FROM weather_candidates wc
)
WHERE rn = 1;


-- ------------------------------------------------------------
-- 6. Calcular indicadores de riesgo
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW gold_fire_risk_source AS
SELECT
    fire_id,
    fire_detection_timestamp,

    fire_latitude,
    fire_longitude,

    forecast_timestamp,

    weather_latitude,
    weather_longitude,
    weather_distance_km,

    temperature_2m,
    relative_humidity_2m,
    precipitation,
    wind_speed_10m,

    brightness,
    frp,

    CASE
        WHEN temperature_2m >= 35 THEN 3
        WHEN temperature_2m >= 30 THEN 2
        WHEN temperature_2m >= 25 THEN 1
        ELSE 0
    END AS temperature_score,

    CASE
        WHEN relative_humidity_2m < 20 THEN 3
        WHEN relative_humidity_2m < 30 THEN 2
        WHEN relative_humidity_2m < 40 THEN 1
        ELSE 0
    END AS humidity_score,

    CASE
        WHEN wind_speed_10m >= 40 THEN 3
        WHEN wind_speed_10m >= 25 THEN 2
        WHEN wind_speed_10m >= 15 THEN 1
        ELSE 0
    END AS wind_score,

    CASE
        WHEN precipitation = 0 THEN 2
        WHEN precipitation < 1 THEN 1
        ELSE 0
    END AS precipitation_score,

    CASE
        WHEN frp >= 100 THEN 3
        WHEN frp >= 50 THEN 2
        WHEN frp >= 10 THEN 1
        ELSE 0
    END AS fire_intensity_score,

    source_fire,
    source_weather
FROM nearest_weather;


-- ------------------------------------------------------------
-- 7. Calcular score final y nivel de riesgo
-- ------------------------------------------------------------

CREATE OR REPLACE TEMP VIEW gold_fire_risk_final AS
SELECT
    *,
    (temperature_score + humidity_score + wind_score + precipitation_score + fire_intensity_score) AS fire_risk_score,

    CASE
        WHEN (temperature_score + humidity_score + wind_score + precipitation_score + fire_intensity_score) >= 12 THEN 'CRITICAL'
        WHEN (temperature_score + humidity_score + wind_score + precipitation_score + fire_intensity_score) >= 9 THEN 'HIGH'
        WHEN (temperature_score + humidity_score + wind_score + precipitation_score + fire_intensity_score) >= 5 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS fire_risk_level,

    current_timestamp() AS updated_at
FROM gold_fire_risk_source;


-- ------------------------------------------------------------
-- 8. Carga incremental mediante MERGE
-- ------------------------------------------------------------

MERGE INTO gold_fire_risk AS target
USING gold_fire_risk_final AS source
ON  target.fire_id = source.fire_id
AND target.fire_detection_timestamp = source.fire_detection_timestamp
AND target.forecast_timestamp = source.forecast_timestamp

WHEN MATCHED THEN
    UPDATE SET
        target.fire_latitude = source.fire_latitude,
        target.fire_longitude = source.fire_longitude,
        target.weather_latitude = source.weather_latitude,
        target.weather_longitude = source.weather_longitude,
        target.weather_distance_km = source.weather_distance_km,
        target.temperature_2m = source.temperature_2m,
        target.relative_humidity_2m = source.relative_humidity_2m,
        target.precipitation = source.precipitation,
        target.wind_speed_10m = source.wind_speed_10m,
        target.brightness = source.brightness,
        target.frp = source.frp,
        target.temperature_score = source.temperature_score,
        target.humidity_score = source.humidity_score,
        target.wind_score = source.wind_score,
        target.precipitation_score = source.precipitation_score,
        target.fire_intensity_score = source.fire_intensity_score,
        target.fire_risk_score = source.fire_risk_score,
        target.fire_risk_level = source.fire_risk_level,
        target.source_fire = source.source_fire,
        target.source_weather = source.source_weather,
        target.updated_at = source.updated_at

WHEN NOT MATCHED THEN
    INSERT (
        fire_id, fire_detection_timestamp, fire_latitude, fire_longitude,
        forecast_timestamp, weather_latitude, weather_longitude, weather_distance_km,
        temperature_2m, relative_humidity_2m, precipitation, wind_speed_10m,
        brightness, frp, temperature_score, humidity_score, wind_score,
        precipitation_score, fire_intensity_score, fire_risk_score, fire_risk_level,
        source_fire, source_weather, updated_at
    )
    VALUES (
        source.fire_id, source.fire_detection_timestamp, source.fire_latitude, source.fire_longitude,
        source.forecast_timestamp, source.weather_latitude, source.weather_longitude, source.weather_distance_km,
        source.temperature_2m, source.relative_humidity_2m, source.precipitation, source.wind_speed_10m,
        source.brightness, source.frp, source.temperature_score, source.humidity_score, source.wind_score,
        source.precipitation_score, source.fire_intensity_score, source.fire_risk_score, source.fire_risk_level,
        source.source_fire, source.source_weather, source.updated_at
    );